<a href="https://colab.research.google.com/github/MariaMuu/Thesis/blob/main/Fine_Tuning_Thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai

In [ ]:
!pip install tiktoken

**Converting CSV to JSONL format**

In [ ]:
import csv
import json

def convert_csv_to_training_format(input_csv, output_file): # change this
    system_message = {
        "role": "system",
        "content": "You are a SPARQL generator for Wikidata. Given a natural language question, return a syntactically correct and executable SPARQL query using the Wikidata ontology. Respond only with the query" #give the basic instruction that apply to every prompt
    }

    with open(input_csv, 'r', encoding='utf-8') as csvfile, \
         open(output_file, 'w', encoding='utf-8') as outfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header

        for row in reader:
            train_cleaned = row[0] # change this
            extracted_json = row[1]

            training_example = { # change this
                "messages": [
                    system_message,
                    {"role": "user", "content": train_cleaned}, # change this
                    {"role": "assistant", "content": extracted_json}
                ]
            }
            outfile.write(json.dumps(training_example) + '\n') # change this

In [ ]:
def convert_csv_to_testing_format(input_csv, output_file): # change this
    system_message = {
        "role": "system",
        "content": "You are a SPARQL generator for Wikidata. Given a natural language question, return a syntactically correct and executable SPARQL query using the Wikidata ontology. Respond only with the query" #give the basic instruction that apply to every prompt
    }

    with open(input_csv, 'r', encoding='utf-8') as csvfile, \
         open(output_file, 'w', encoding='utf-8') as outfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header

        for row in reader:
            test_cleaned = row[0] # change this
            extracted_json = row[1]

            testing_example = { #change this
                "messages": [
                    system_message,
                    {"role": "user", "content": test_cleaned}, # change this
                    {"role": "assistant", "content": extracted_json}
                ]
            }
            outfile.write(json.dumps(testing_example) + '\n') # change this

**Creating Training and Validation Sets**

In [ ]:
# Prepare training data
convert_csv_to_training_format("train_cleaned.csv", "sparql_thesis_training_data.jsonl") # change

# Prepare validation data
convert_csv_to_testing_format("test_cleaned.csv", "sparql_thesis_validation_data.jsonl") # change

# Setting Up and Starting the Fine-Tuning Process

**Initial Setup with OpenAI**

In [ ]:
from openai import OpenAI
from time import sleep
from google.colab import userdata

#Access the API key from colab secrets.
api_key = userdata.get('OPENAI_API_KEY')

if not api_key:
    # This will happen if the secret is not set or notebook access is off
    raise ValueError("OpenAI API key not found in Colab Secrets. Please add it named 'OPENAI_API_KEY'.")

# Initialize OpenAI client
client = OpenAI(api_key = api_key)


**Step 1: Uploading Training Files**

In [ ]:
def upload_training_file(file_path):
    """Upload training file to OpenAI"""
    with open(file_path, "rb") as file:
        response = client.files.create(
            file=file,
            purpose="fine-tune"
        )
        return response.id

# Upload both training and validation files
training_file_id = upload_training_file("sparql_thesis_training_data.jsonl") # change
validation_file_id = upload_training_file("sparql_thesis_validation_data.jsonl") # change

**Calculating the number of tokens and the fine-tuning's cost estimated.**

by https://github.com/superasperatus/openai-fine-tuning-cost-estimates/blob/main/LICENSE and changed a bit.

In [ ]:
import json
import tiktoken  # Make sure that tiktoken is installed

def count_tokens_in_jsonl(jsonl_file_path, encoding_name='cl100k_base'):
    """
    This function counts the number of tokens for the serialized JSON object on each line of a JSONL file.

    Parameters:
    jsonl_file_path (str): Path to the JSONL file.
    encoding_name (str): The encoding to use for tokenization (default is 'cl100k_base').

    Returns:
    int: The total number of tokens across all JSON objects in the file.
    """
    total_token_count = 0
    encoding = tiktoken.get_encoding(encoding_name)

    with open(jsonl_file_path, 'r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, 1):
            try:
                # Parse JSON to ensure it's valid
                json_object = json.loads(line)

                # Serialize JSON object back to string
                text_content = json.dumps(json_object, ensure_ascii=False)

                # Tokenize and count
                token_count = len(encoding.encode(text_content))
                total_token_count += token_count

                # Optional: Print the number of tokens in this line
                print(f"Line {line_number}: {token_count} tokens")

            except json.JSONDecodeError as e:
                print(f"Error parsing JSON on line {line_number}: {e}")

    return total_token_count

# Usage:
jsonl_file_path = '/content/sparql_thesis_training_data.jsonl'
total_tokens = count_tokens_in_jsonl(jsonl_file_path)
print(f"Total tokens across entire file: {total_tokens}")

# Price for gpt-4.1-mini-2025-04-14 training is $5.00 per 1 million tokens
price_per_million_tokens = 5.00
tokens_per_million = 1000000

# Calculate the number of millions of tokens
millions_of_tokens = total_tokens / tokens_per_million

# Assume 4 training epochs
num_epochs = 5

# Calculate the estimated cost
cost_estimates = millions_of_tokens * price_per_million_tokens * num_epochs

print(f"Estimated costs for fine-tuning this jsonl with gpt-4.1-mini-2025-04-14 ({num_epochs} epochs) is: $ {cost_estimates}")

Line 1: 133 tokens
Line 2: 129 tokens
Line 3: 158 tokens
Line 4: 123 tokens
Line 5: 125 tokens
Line 6: 118 tokens
Line 7: 117 tokens
Line 8: 116 tokens
Line 9: 157 tokens
Line 10: 120 tokens
Line 11: 123 tokens
Line 12: 130 tokens
Line 13: 116 tokens
Line 14: 110 tokens
Line 15: 160 tokens
Line 16: 125 tokens
Line 17: 155 tokens
Line 18: 130 tokens
Line 19: 139 tokens
Line 20: 126 tokens
Line 21: 156 tokens
Line 22: 164 tokens
Line 23: 160 tokens
Line 24: 118 tokens
Line 25: 129 tokens
Line 26: 132 tokens
Line 27: 122 tokens
Line 28: 115 tokens
Line 29: 125 tokens
Line 30: 160 tokens
Line 31: 136 tokens
Line 32: 124 tokens
Line 33: 130 tokens
Line 34: 120 tokens
Line 35: 159 tokens
Line 36: 127 tokens
Line 37: 115 tokens
Line 38: 113 tokens
Line 39: 134 tokens
Line 40: 117 tokens
Line 41: 139 tokens
Line 42: 121 tokens
Line 43: 120 tokens
Line 44: 129 tokens
Line 45: 129 tokens
Line 46: 117 tokens
Line 47: 121 tokens
Line 48: 129 tokens
Line 49: 131 tokens
Line 50: 123 tokens
Line 51: 

**Step 2: Creating a Fine-Tuning Job**

In [ ]:
!head -n 5 /content/sparql_thesis_training_data.jsonl
!head -n 5 /content/sparql_thesis_validation_data.jsonl

{"messages": [{"role": "system", "content": "You are a SPARQL generator for Wikidata. Given a natural language question, return a syntactically correct and executable SPARQL query using the Wikidata ontology. Respond only with the query"}, {"role": "user", "content": "Who was Galileo Galilei's employer in 1592?"}, {"role": "assistant", "content": "SELECT ?obj WHERE { wd:Q307 p:P108 ?s . ?s ps:P108 ?obj . ?s pq:P580 ?x filter(contains(YEAR(?x),'1592')) }"}]}
{"messages": [{"role": "system", "content": "You are a SPARQL generator for Wikidata. Given a natural language question, return a syntactically correct and executable SPARQL query using the Wikidata ontology. Respond only with the query"}, {"role": "user", "content": "When did Helen Caldicott receive the award for Victorian Honour Roll of Women?"}, {"role": "assistant", "content": "SELECT ?value WHERE { wd:Q431092 p:P166 ?s . ?s ps:P166 wd:Q7927224 . ?s pq:P585 ?value}"}]}
{"messages": [{"role": "system", "content": "You are a SPARQ

In [ ]:
def create_fine_tuning_job(training_file_id, validation_file_id=None, model="gpt-4.1-mini-2025-04-14"): # change model
    """Create a fine-tuning job"""
    response = client.fine_tuning.jobs.create(
        training_file=training_file_id,
        validation_file=validation_file_id,
        model=model,
        hyperparameters={
            "n_epochs": 5,
            "learning_rate_multiplier": 0.2
            }
    )
    return response.id

model = "gpt-4.1-mini-2025-04-14"

# Start the fine-tuning job
job_id = create_fine_tuning_job(training_file_id, validation_file_id, model)

**Step 3: Monitoring Training Progress**

In [ ]:
def monitor_job(job_id):
    """Monitor fine-tuning job progress"""
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        print(f"Status: {job.status}")

        if job.status in ["succeeded", "failed"]:
            return job

        # List latest events
        events = client.fine_tuning.jobs.list_events(
            fine_tuning_job_id=job_id,
            limit=5
        )
        for event in events.data:
            print(f"Event: {event.message}")

        sleep(30)  # Check every 30 seconds

# Monitor the job until completion
job = monitor_job(job_id)
if job.status == "succeeded":
    fine_tuned_model = job.fine_tuned_model
    print(f"Fine-tuned model ID: {fine_tuned_model}")
else:
    print("Fine-tuning failed.")

Status: validating_files
Event: Validating training file: file-3exYt8er5h8CYp5zKQVEm4 and validation file: file-69FyzTTm3KUNQj4ohHz8MR
Event: Created fine-tuning job: ftjob-7ayIN4c71X8L5CpVvxUJCOst
Status: validating_files
Event: Validating training file: file-3exYt8er5h8CYp5zKQVEm4 and validation file: file-69FyzTTm3KUNQj4ohHz8MR
Event: Created fine-tuning job: ftjob-7ayIN4c71X8L5CpVvxUJCOst
Status: validating_files
Event: Validating training file: file-3exYt8er5h8CYp5zKQVEm4 and validation file: file-69FyzTTm3KUNQj4ohHz8MR
Event: Created fine-tuning job: ftjob-7ayIN4c71X8L5CpVvxUJCOst
Status: validating_files
Event: Validating training file: file-3exYt8er5h8CYp5zKQVEm4 and validation file: file-69FyzTTm3KUNQj4ohHz8MR
Event: Created fine-tuning job: ftjob-7ayIN4c71X8L5CpVvxUJCOst
Status: validating_files
Event: Validating training file: file-3exYt8er5h8CYp5zKQVEm4 and validation file: file-69FyzTTm3KUNQj4ohHz8MR
Event: Created fine-tuning job: ftjob-7ayIN4c71X8L5CpVvxUJCOst
Status: va

# **Testing and Using Your Fine-Tuned Model**

**Making Predictions with Your Model**

In [ ]:
def test_model(model_id, test_input):
    """Test the fine-tuned model"""
    completion = client.chat.completions.create(
        model=model_id,
        messages=[
            {
                "role": "system",
                "content": "You are a SPARQL generator for Wikidata. Given a natural language question, return a syntactically correct and executable SPARQL query using the Wikidata ontology. Respond only with the query" # change this
            },
            {"role": "user", "content": test_input}
        ]
    )
    return completion.choices[0].message

**Let's try it with a new medical report:**

In [ ]:
# Test input
test_report = """"What is a famous painting of a famous artist?"""

# Get prediction
result = test_model(fine_tuned_model, test_report)

print(result.content)

SELECT ?answer WHERE { wd:Q230018 wdt:P170 ?X . ?X wdt:P800 ?answer}
